# 03 — Offline training (CatBoost)

Fit a frozen `p_offline` classifier. The streaming pipeline references it via `spec.wagie.model.catboost_path` — at runtime `wagie.pipeline.FrozenCatBoostPredictor` loads the .cbm and runs forward only.

Online training is a different concept entirely: ARF and Mondrian-ACI learn *streamingly* inside the engine from the LabelBuffer. This notebook is concerned only with the offline layer.

**No bespoke training loop is part of the package.** Use this notebook (or any equivalent) to produce a `.cbm`; the package consumes it through the spec.

In [ ]:
from pathlib import Path
import numpy as np
import polars as pl

src = Path("data/cleansed_data/btcusdt_20m_labeled.parquet")
assert src.is_file(), f"run 02_feature_build first; missing {src}"
df = pl.read_parquet(src).filter(pl.col("label").is_not_null())
print(f"labelled rows: {len(df)}")
df.head(3)

Build a small numeric feature matrix from the OHLC columns. In practice you'd point CatBoost at the streaming features (run the engine once with `capture_audit=True` to dump them, or compute them offline with the same catalog), but the goal here is a tractable demo.

In [ ]:
import polars as pl

feat = df.with_columns(
    log_close=pl.col("close").log(),
    log_range=(pl.col("high") / pl.col("low")).log(),
    ret_1=(pl.col("close") / pl.col("close").shift(1)).log(),
    ret_5=(pl.col("close") / pl.col("close").shift(5)).log(),
    parkinson=((pl.col("high") / pl.col("low")).log()) ** 2 / (4.0 * np.log(2.0)),
).drop_nulls()

FEATURES = ["log_close", "log_range", "ret_1", "ret_5", "parkinson"]
X = feat.select(FEATURES).to_numpy()
y = feat["label"].to_numpy().astype(int)
print(f"X.shape={X.shape}, y.mean={y.mean():.3f}")

Train/val split — chronological, 60/40. CatBoost with langevin + ordered + balanced (the same Phase-A defaults the repo's been using).

In [ ]:
from catboost import CatBoostClassifier, Pool

n_train = int(len(X) * 0.6)
Xtr, ytr = X[:n_train], y[:n_train]
Xva, yva = X[n_train:], y[n_train:]

model = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=6,
    auto_class_weights="SqrtBalanced",
    boosting_type="Ordered",
    bootstrap_type="MVS",
    langevin=True,
    diffusion_temperature=10000,
    eval_metric="AUC",
    random_seed=42,
    verbose=0,
)
model.fit(Pool(Xtr, ytr), eval_set=Pool(Xva, yva), use_best_model=True)
p_va = model.predict_proba(Xva)[:, 1]
print(f"trees used: {model.tree_count_}")

Evaluate via the package's MetricsBattery — calibration metrics lead, ranking is diagnostic:

In [ ]:
from wagie.metrics import (
    brier_score, expected_calibration_error,
    roc_auc, pr_auc,
)

print(f"Brier:  {brier_score(yva, p_va):.5f}")
print(f"ECE:    {expected_calibration_error(yva, p_va):.5f}")
print(f"AUC:    {roc_auc(yva, p_va):.4f}")
print(f"AP:     {pr_auc(yva, p_va):.4f}")

In [ ]:
import json
from pathlib import Path

out_dir = Path("artifacts/offline_model")
out_dir.mkdir(parents=True, exist_ok=True)

model_path = out_dir / "model.cbm"
model.save_model(str(model_path))
(out_dir / "selected_features.json").write_text(
    json.dumps({"features": FEATURES}, indent=2),
    encoding="utf-8",
)
print(f"saved → {model_path}")

Wire it into a spec for **04_online_eval**:

```yaml
wagie:
  model:
    catboost_path: artifacts/offline_model/model.cbm
    selected_features_path: artifacts/offline_model/selected_features.json
```